# Projeto (Seção 11) — solução do instrutor

## Especificação

Foi aqui que paramos no código da seção anterior:

In [1]:
countries = {
    "australia": ("Australia", "AUS"),
    "canada": ("Canada", "CAN"),
    "china": ("China", "CHN"),
    "france": ("France", "FRA"),
    "germany": ("Germany", "DEU"),
    "india": ("India", "IND"),
    "mexico": ("Mexico", "MEX"),
    "norway": ("Norway", "NOR"),
    "pakistan": ("Pakistan", "PAK"),
    "san marino": ("San Marino", "SMR"),
    "sanmarino": ("San Marino", "SMR"),
    "spain": ("Spain", "ESP"),
    "sweden": ("Sweden", "SWE"),
    "united kingdom": ("United Kingdom", "GBR"),
    "uk": ("United Kingdom", "GBR"),
    "great britain": ("United Kingdom", "GBR"),
    "britain": ("United Kingdom", "GBR"),
    "us": ("United States of America", "USA"),
    "united states": ("United States of America", "USA"),
    "usa": ("United States of America", "USA"),
}
valid_country_names = sorted(countries.keys())

def lookup_country(name: str) -> tuple[str, str]:
    name = name.strip().casefold()
    
    try:
        return countries[name]
    except KeyError:
        raise ValueError(
            "Unknown country name. "
            f"Country name must be one of: {','.join(valid_country_names)}"
        )

country_code_lookup = {
    name: code
    for name, code in countries.values()
}

In [2]:
from datetime import date
from enum import Enum
from functools import cached_property
from typing import Annotated, TypeVar
from uuid import uuid4
from pydantic import (
    AfterValidator, 
    BaseModel, 
    ConfigDict, 
    computed_field,
    Field, 
    field_serializer, 
    field_validator,
    PlainSerializer,
    UUID4,
    ValidationInfo,
)
from pydantic.alias_generators import to_camel


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


T = TypeVar('T')
BoundedString = Annotated[str, Field(min_length=2, max_length=50)]
BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]
Country = Annotated[str, AfterValidator(lambda name: lookup_country(name)[0])]


def serialize_date(value: date) -> str:
        return value.strftime("%Y/%m/%d")


CustomDate = Annotated[
    date, 
    PlainSerializer(serialize_date, when_used="json-unless-none")
]

class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

    id_: UUID4 | None = Field(alias="id", default_factory=uuid4) 
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = Field(default=False, repr=False)
    manufactured_date: CustomDate = Field(
        validation_alias="completionDate", 
        ge=date(1980, 1, 1), 
        repr=False
    )
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD", 
        serialization_alias="baseMSRPUSD",
        repr=False,
    )
    top_features: BoundedList[BoundedString] | None = Field(default=None, repr=False)
    vin: BoundedString = Field(repr=False)
    number_of_doors: int = Field(
        default=4, 
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
        repr=False,
    )
    registration_country: Country | None = Field(default=None, repr=False)

    @computed_field(repr=False)
    @cached_property
    def registration_country_code(self) -> str:
        return country_code_lookup[self.registration_country] 
    
    registration_date: CustomDate | None = Field(default=None, repr=False)
    license_plate: BoundedString | None = Field(default=None, repr=False)
        
    @field_validator("registration_date")
    @classmethod
    def validate_registration_date(cls, value:date, values: ValidationInfo):
        data = values.data
        if "manufactured_date" in data and data["manufactured_date"] > value:
            raise ValueError("Automobile cannot be registered prior to manufacture date.")
        return value

Agora vamos refatorar o modelo e colocar as informações do país de registro num modelo separado, que depois incluímos no modelo principal por **composição**.

Antes disso, queremos aplicar a mesma configuração a todos os modelos.

Então, primeiro crie um modelo base customizado (chame-o de `CamelBaseModel`) com a mesma configuração do `Automobile` atual.

Depois, use **herança** para remover a configuração de `Automobile`, que passa a herdá-la de `CamelBaseModel`.

Em seguida, crie um modelo separado (também baseado em `CamelBaseModel`) para onde devem ir estes campos de `Automobile` (incluindo validadores, campos calculados etc.). Aproveite para limpar as configurações de representação desse modelo ao copiar.

- implemente o campo `registration_country` no novo modelo — mas renomeie-o simplesmente para `name`;
- implemente o campo calculado `registration_country_code` no novo modelo — mas renomeie-o simplesmente para `code3`.

Com esse modelo definido (e removidos `registration_country` e `registration_country_code` de `Automobile`), crie um único campo substituto, também chamado `registration_country`, mas agora com o submodelo como tipo. Claro, o campo continua com default `None`.

## Solução

Primeiro, o modelo base customizado:

In [3]:
class CamelBaseModel(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

Depois, redefinimos a classe `Automobile` assim:

In [4]:
class Automobile(CamelBaseModel):
    id_: UUID4 | None = Field(alias="id", default_factory=uuid4) 
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = Field(default=False, repr=False)
    manufactured_date: CustomDate = Field(
        validation_alias="completionDate", 
        ge=date(1980, 1, 1), 
        repr=False
    )
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD", 
        serialization_alias="baseMSRPUSD",
        repr=False,
    )
    top_features: BoundedList[BoundedString] | None = Field(default=None, repr=False)
    vin: BoundedString = Field(repr=False)
    number_of_doors: int = Field(
        default=4, 
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
        repr=False,
    )
    registration_country: Country | None = Field(default=None, repr=False)

    @computed_field(repr=False)
    @cached_property
    def registration_country_code(self) -> str:
        return country_code_lookup[self.registration_country] 
    
    registration_date: CustomDate | None = Field(default=None, repr=False)
    license_plate: BoundedString | None = Field(default=None, repr=False)
        
    @field_validator("registration_date")
    @classmethod
    def validate_registration_date(cls, value:date, values: ValidationInfo):
        data = values.data
        if "manufactured_date" in data and data["manufactured_date"] > value:
            raise ValueError("Automobile cannot be registered prior to manufacture date.")
        return value

Em seguida, um novo modelo para o país de registro:

In [5]:
class RegistrationCountry(CamelBaseModel):
    name: Country | None = Field(default=None)

    @computed_field
    @cached_property
    def code3(self) -> str:
        return country_code_lookup[self.name]

E agora, com composição, refatoramos os campos de `Automobile`:

In [6]:
class Automobile(CamelBaseModel):
    id_: UUID4 | None = Field(alias="id", default_factory=uuid4) 
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = Field(default=False, repr=False)
    manufactured_date: CustomDate = Field(
        validation_alias="completionDate", 
        ge=date(1980, 1, 1), 
        repr=False
    )
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD", 
        serialization_alias="baseMSRPUSD",
        repr=False,
    )
    top_features: BoundedList[BoundedString] | None = Field(default=None, repr=False)
    vin: BoundedString = Field(repr=False)
    number_of_doors: int = Field(
        default=4, 
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
        repr=False,
    )
    registration_country: RegistrationCountry | None = Field(default=None, repr=False)
    registration_date: CustomDate | None = Field(default=None, repr=False)
    license_plate: BoundedString | None = Field(default=None, repr=False)
        
    @field_validator("registration_date")
    @classmethod
    def validate_registration_date(cls, value:date, values: ValidationInfo):
        data = values.data
        if "manufactured_date" in data and data["manufactured_date"] > value:
            raise ValueError("Automobile cannot be registered prior to manufacture date.")
        return value

Por fim, desserializamos alguns dados:

In [7]:
from uuid import UUID

data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": {"name": "us"},
    "registrationDate": "2023-06-01",
    "licensePlate": "AAA-BBB"
}

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': {
        'name': 'United States of America', 
        'code3': 'USA'
    },
    'registrationDate': date(2023, 6, 1),
    'licensePlate': 'AAA-BBB'
}

In [8]:
car = Automobile.model_validate(data)
print(f"car: {car!r}")

car: Automobile(id_=UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), manufacturer='BMW', series_name='M4 Competition xDrive', type_=<AutomobileType.convertible: 'Convertible'>)


In [9]:
print(f"car.registration_country: {car.registration_country!r}")

car.registration_country: RegistrationCountry(name='United States of America', code3='USA')


E serializamos:

> 📝 **Nota de revisão:** o original diz "desserializar" aqui, mas a célula abaixo **serializa**.

In [10]:
print(f"car.model_dump(by_alias=True): {car.model_dump(by_alias=True)!r}")

car.model_dump(by_alias=True): {'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), 'manufacturer': 'BMW', 'seriesName': 'M4 Competition xDrive', 'type': <AutomobileType.convertible: 'Convertible'>, 'isElectric': False, 'manufacturedDate': datetime.date(2023, 1, 1), 'baseMSRPUSD': 93300.0, 'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'], 'vin': '1234567890', 'numberOfDoors': 2, 'registrationCountry': {'name': 'United States of America', 'code3': 'USA'}, 'registrationDate': datetime.date(2023, 6, 1), 'licensePlate': 'AAA-BBB'}


In [11]:
assert car.model_dump(by_alias=True) == expected_serialized_by_alias

> 📝 **Nota de revisão — os dois bugs da seção 9 migraram para o submodelo:**
>
> 1. `name: Country | None` permite um país **sem nome** (`"registrationCountry": {}`), e aí `code3` faz `country_code_lookup[None]` → `KeyError` em qualquer `model_dump()`. Se o submodelo existe, o nome deveria ser obrigatório (a ausência de país já é expressa por `registration_country=None` no `Automobile`).
> 2. `code3` é `cached_property`, mas `name` continua mutável (`validate_assignment=True` herdado, sem `frozen`): trocar o nome deixa o código antigo no cache.
>
> Também continua o `TypeError` da seção 8 no validador de `registration_date`. Provas:

In [12]:
try:
    Automobile.model_validate(data | {"registrationCountry": {}}).model_dump()
except Exception as ex:
    print(f"1) país sem nome: {type(ex).__name__}: {ex}")

car = Automobile.model_validate(data)
car.registration_country.code3                       # cacheia 'USA'
car.registration_country.name = "france"
print(f"2) nome trocado para {car.registration_country.name}, code3 continua {car.registration_country.code3}")

try:
    Automobile.model_validate({k: v for k, v in data.items() if k != "registrationDate"})
except Exception as ex:
    print(f"3) sem registrationDate: {type(ex).__name__}: {ex}")

1) país sem nome: KeyError: None
2) nome trocado para France, code3 continua USA
3) sem registrationDate: TypeError: '>' not supported between instances of 'datetime.date' and 'NoneType'
